# Design Rewards That Don't Train Hallucinations

**Goal:** Understand how reward function design shapes LLM behavior.

**The Problem:** If you optimize only for user satisfaction, you'll train a system that hallucinates confidently. If you optimize only for brevity, you'll train a system that gives shallow answers.

**The Solution:** Composite rewards with guardrails — primary metric + penalties for bad behaviors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Simulate a Commodity Analysis Bot

We have 4 prompts with different characteristics:
1. **Confident & Detailed** - Users love it, but hallucinates 40% of the time
2. **Cautious & Accurate** - Boring but correct, low hallucination rate
3. **Concise & Direct** - Short answers, moderate accuracy
4. **Balanced** - Good middle ground

We'll test 3 different reward designs and see which prompts they favor.

In [ ]:
# Prompt characteristics (simulated)
prompts = [
    "Confident & Detailed",
    "Cautious & Accurate", 
    "Concise & Direct",
    "Balanced"
]

# Characteristics for each prompt
characteristics = pd.DataFrame({
    'user_satisfaction': [0.90, 0.60, 0.70, 0.80],  # How much users like it
    'task_completion': [0.70, 0.85, 0.75, 0.80],    # Does it answer the question?
    'accuracy': [0.55, 0.95, 0.75, 0.85],           # Factual correctness
    'hallucination_rate': [0.40, 0.05, 0.20, 0.10], # Fraction of unsupported claims
    'length': [250, 120, 60, 150],                   # Average word count
}, index=prompts)

print("Prompt Characteristics:")
print(characteristics.round(2))
print("\nKey insight: Confident & Detailed has highest user satisfaction but also highest hallucination rate!")

## Reward Design A: User Satisfaction Only

**Bad reward:**
```python
reward = user_satisfaction
```

This is what happens when you optimize for thumbs-up clicks without considering accuracy.

In [ ]:
def reward_A(prompt_idx):
    """Reward A: User satisfaction only."""
    satisfaction = characteristics['user_satisfaction'].iloc[prompt_idx]
    noise = np.random.normal(0, 0.05)
    return np.clip(satisfaction + noise, 0, 1)

# Run bandit with Reward A
class SimpleThompson:
    def __init__(self, n_arms):
        self.alpha = np.ones(n_arms)
        self.beta = np.ones(n_arms)
    
    def select(self):
        samples = [np.random.beta(a, b) for a, b in zip(self.alpha, self.beta)]
        return np.argmax(samples)
    
    def update(self, idx, reward):
        if reward > 0.7:
            self.alpha[idx] += 1
        else:
            self.beta[idx] += 1

# Simulate
bandit_A = SimpleThompson(n_arms=4)
history_A = []

for t in range(300):
    idx = bandit_A.select()
    reward = reward_A(idx)
    bandit_A.update(idx, reward)
    history_A.append({'step': t, 'prompt': idx, 'reward': reward})

history_A_df = pd.DataFrame(history_A)

# Results
print("\nReward A Results (User Satisfaction Only):")
print("\nPrompt selection frequency (last 100 requests):")
final_selections_A = history_A_df.tail(100)['prompt'].value_counts(normalize=True).sort_index()
for idx, freq in final_selections_A.items():
    prompt_name = prompts[idx]
    halluc_rate = characteristics['hallucination_rate'].iloc[idx]
    print(f"  {prompt_name}: {freq:.1%} (hallucination rate: {halluc_rate:.1%})")

print("\n⚠️  Problem: System learned to use Confident & Detailed (highest hallucination rate!)")

## Reward Design B: Accuracy Only

**Opposite extreme:**
```python
reward = accuracy
```

This optimizes for correctness but ignores whether it's actually useful.

In [ ]:
def reward_B(prompt_idx):
    """Reward B: Accuracy only."""
    accuracy = characteristics['accuracy'].iloc[prompt_idx]
    noise = np.random.normal(0, 0.05)
    return np.clip(accuracy + noise, 0, 1)

# Simulate
bandit_B = SimpleThompson(n_arms=4)
history_B = []

for t in range(300):
    idx = bandit_B.select()
    reward = reward_B(idx)
    bandit_B.update(idx, reward)
    history_B.append({'step': t, 'prompt': idx, 'reward': reward})

history_B_df = pd.DataFrame(history_B)

# Results
print("\nReward B Results (Accuracy Only):")
print("\nPrompt selection frequency (last 100 requests):")
final_selections_B = history_B_df.tail(100)['prompt'].value_counts(normalize=True).sort_index()
for idx, freq in final_selections_B.items():
    prompt_name = prompts[idx]
    satisfaction = characteristics['user_satisfaction'].iloc[idx]
    print(f"  {prompt_name}: {freq:.1%} (user satisfaction: {satisfaction:.1%})")

print("\n⚠️  Problem: System learned Cautious & Accurate (boring, low satisfaction)")

## Reward Design C: Composite with Guardrails

**The right approach:**
```python
reward = primary_metric + guardrails
reward = 0.5 * task_completion + 0.3 * accuracy - 0.5 * hallucination_penalty
```

This balances usefulness (task completion), correctness (accuracy), and safety (hallucination penalty).

In [ ]:
def reward_C(prompt_idx):
    """Reward C: Composite with guardrails."""
    task_comp = characteristics['task_completion'].iloc[prompt_idx]
    accuracy = characteristics['accuracy'].iloc[prompt_idx]
    halluc_rate = characteristics['hallucination_rate'].iloc[prompt_idx]
    
    # Primary metrics
    primary = 0.5 * task_comp + 0.3 * accuracy
    
    # Guardrail: hallucination penalty
    halluc_penalty = -0.5 * halluc_rate
    
    # Composite
    reward = primary + halluc_penalty
    
    noise = np.random.normal(0, 0.05)
    return np.clip(reward + noise, 0, 1)

# Simulate
bandit_C = SimpleThompson(n_arms=4)
history_C = []

for t in range(300):
    idx = bandit_C.select()
    reward = reward_C(idx)
    bandit_C.update(idx, reward)
    history_C.append({'step': t, 'prompt': idx, 'reward': reward})

history_C_df = pd.DataFrame(history_C)

# Results
print("\nReward C Results (Composite: Task Completion + Accuracy - Hallucination):")
print("\nPrompt selection frequency (last 100 requests):")
final_selections_C = history_C_df.tail(100)['prompt'].value_counts(normalize=True).sort_index()
for idx, freq in final_selections_C.items():
    prompt_name = prompts[idx]
    halluc_rate = characteristics['hallucination_rate'].iloc[idx]
    task_comp = characteristics['task_completion'].iloc[idx]
    print(f"  {prompt_name}: {freq:.1%} (task: {task_comp:.1%}, halluc: {halluc_rate:.1%})")

print("\n✅ Success: System learned Balanced (good task completion, low hallucinations)")

## Visualize: Which Prompts Win Under Each Reward?

In [ ]:
# Compile selection frequencies
selection_data = []
for idx in range(4):
    selection_data.append({
        'Prompt': prompts[idx],
        'Reward A\n(Satisfaction)': final_selections_A.get(idx, 0),
        'Reward B\n(Accuracy)': final_selections_B.get(idx, 0),
        'Reward C\n(Composite)': final_selections_C.get(idx, 0),
    })

selection_df = pd.DataFrame(selection_data)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(prompts))
width = 0.25

ax.bar(x - width, selection_df['Reward A\n(Satisfaction)'], width, 
       label='Reward A (Satisfaction)', color='#ff6b6b', alpha=0.8)
ax.bar(x, selection_df['Reward B\n(Accuracy)'], width, 
       label='Reward B (Accuracy)', color='#4ecdc4', alpha=0.8)
ax.bar(x + width, selection_df['Reward C\n(Composite)'], width, 
       label='Reward C (Composite)', color='#95e1d3', alpha=0.8)

ax.set_xlabel('Prompt', fontsize=12)
ax.set_ylabel('Selection Frequency (last 100)', fontsize=12)
ax.set_title('Which Prompts Win Under Different Reward Functions?', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(prompts, rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Insight:")
print("  Reward A → Confident & Detailed (high hallucinations)")
print("  Reward B → Cautious & Accurate (boring but correct)")
print("  Reward C → Balanced (good trade-off)")

## Show How Reward A Trains Hallucinations

Let's track the **average hallucination rate** of selected prompts over time for each reward design.

In [ ]:
# Calculate rolling average hallucination rate
def calc_halluc_rate_over_time(history_df, window=20):
    """Calculate average hallucination rate of selected prompts."""
    halluc_rates = []
    for i in range(len(history_df)):
        prompt_idx = history_df.iloc[i]['prompt']
        halluc_rate = characteristics['hallucination_rate'].iloc[prompt_idx]
        halluc_rates.append(halluc_rate)
    
    # Rolling average
    return pd.Series(halluc_rates).rolling(window=window).mean()

halluc_A = calc_halluc_rate_over_time(history_A_df)
halluc_B = calc_halluc_rate_over_time(history_B_df)
halluc_C = calc_halluc_rate_over_time(history_C_df)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(halluc_A.values, label='Reward A (Satisfaction Only)', linewidth=2.5, color='#ff6b6b')
ax.plot(halluc_B.values, label='Reward B (Accuracy Only)', linewidth=2.5, color='#4ecdc4')
ax.plot(halluc_C.values, label='Reward C (Composite)', linewidth=2.5, color='#95e1d3')

ax.axhline(y=0.15, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Acceptable threshold (15%)')

ax.set_xlabel('Request Number', fontsize=12)
ax.set_ylabel('Average Hallucination Rate (rolling)', fontsize=12)
ax.set_title('Reward A Trains Hallucinations, Reward C Prevents Them', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFinal hallucination rates (last 50 requests):")
print(f"  Reward A: {halluc_A.tail(50).mean():.1%} ← DANGEROUS")
print(f"  Reward B: {halluc_B.tail(50).mean():.1%} ← Safe but boring")
print(f"  Reward C: {halluc_C.tail(50).mean():.1%} ← Balanced")

## Implement Guardrails: Reject High Hallucination Prompts

**Hard constraint:** Never select a prompt if its estimated hallucination rate > 25%.

In [ ]:
class ThompsonWithGuardrails:
    def __init__(self, n_arms):
        self.alpha = np.ones(n_arms)
        self.beta = np.ones(n_arms)
        self.halluc_counts = np.zeros(n_arms)  # Track hallucinations
        self.total_uses = np.zeros(n_arms)
    
    def select(self):
        """Select, but reject if hallucination rate estimate > 25%."""
        for _ in range(100):  # Try up to 100 times
            samples = [np.random.beta(a, b) for a, b in zip(self.alpha, self.beta)]
            idx = np.argmax(samples)
            
            # Check hallucination rate estimate
            if self.total_uses[idx] > 5:  # Need some data
                halluc_rate_est = self.halluc_counts[idx] / self.total_uses[idx]
                if halluc_rate_est > 0.25:
                    continue  # Reject, try again
            
            return idx
        
        # Fallback: select least-used arm
        return np.argmin(self.total_uses)
    
    def update(self, idx, reward, hallucinated):
        """Update with reward and hallucination flag."""
        if reward > 0.7:
            self.alpha[idx] += 1
        else:
            self.beta[idx] += 1
        
        self.total_uses[idx] += 1
        if hallucinated:
            self.halluc_counts[idx] += 1

# Simulate with guardrails
bandit_guarded = ThompsonWithGuardrails(n_arms=4)
history_guarded = []

for t in range(300):
    idx = bandit_guarded.select()
    reward = reward_A(idx)  # Using Reward A (satisfaction)
    
    # Simulate hallucination detection
    halluc_rate = characteristics['hallucination_rate'].iloc[idx]
    hallucinated = np.random.rand() < halluc_rate
    
    bandit_guarded.update(idx, reward, hallucinated)
    history_guarded.append({'step': t, 'prompt': idx, 'reward': reward})

history_guarded_df = pd.DataFrame(history_guarded)

print("\nWith Guardrails (Hallucination Rate Constraint):")
print("\nPrompt selection frequency (last 100 requests):")
final_selections_guarded = history_guarded_df.tail(100)['prompt'].value_counts(normalize=True).sort_index()
for idx, freq in final_selections_guarded.items():
    prompt_name = prompts[idx]
    halluc_rate = characteristics['hallucination_rate'].iloc[idx]
    print(f"  {prompt_name}: {freq:.1%} (hallucination rate: {halluc_rate:.1%})")

print("\n✅ Guardrail prevented selection of Confident & Detailed (40% hallucination rate)")

## Modify This: Design Your Own Composite Reward

**Your turn:** Design a reward function for a commodity trading signal system.

**Available metrics:**
- `task_completion`: Did it generate a signal?
- `accuracy`: Was the signal directionally correct?
- `hallucination_rate`: Unsupported claims
- `user_satisfaction`: Did the trader find it useful?

**Questions to consider:**
1. What should be the primary metric?
2. How heavily should you penalize hallucinations?
3. Should satisfaction matter at all?

Implement `reward_D()` below and test it.

In [ ]:
def reward_D(prompt_idx):
    """YOUR CUSTOM REWARD FUNCTION."""
    # Extract characteristics
    task_comp = characteristics['task_completion'].iloc[prompt_idx]
    accuracy = characteristics['accuracy'].iloc[prompt_idx]
    halluc_rate = characteristics['hallucination_rate'].iloc[prompt_idx]
    satisfaction = characteristics['user_satisfaction'].iloc[prompt_idx]
    
    # YOUR DESIGN HERE
    # Example:
    reward = (
        0.6 * accuracy +              # Primary: correct signals
        0.3 * task_completion +       # Secondary: actually produces signals
        -1.0 * halluc_rate +          # Heavy penalty for hallucinations
        0.1 * satisfaction            # Small bonus for user happiness
    )
    
    noise = np.random.normal(0, 0.05)
    return np.clip(reward + noise, 0, 1)

# Test your reward
bandit_D = SimpleThompson(n_arms=4)
history_D = []

for t in range(300):
    idx = bandit_D.select()
    reward = reward_D(idx)
    bandit_D.update(idx, reward)
    history_D.append({'step': t, 'prompt': idx, 'reward': reward})

history_D_df = pd.DataFrame(history_D)

print("\nYour Custom Reward Results:")
print("\nPrompt selection frequency (last 100 requests):")
final_selections_D = history_D_df.tail(100)['prompt'].value_counts(normalize=True).sort_index()
for idx, freq in final_selections_D.items():
    print(f"  {prompts[idx]}: {freq:.1%}")

print("\nDoes this match your expectations? If not, adjust the weights above!")

## Summary: Reward Design Principles

**What we learned:**

1. **User satisfaction alone trains hallucinations** — users like confident answers, even if wrong
2. **Accuracy alone produces boring outputs** — technically correct but not useful
3. **Composite rewards work best** — primary metric + guardrails
4. **Hallucination penalty must be severe** — otherwise the bandit will learn to hallucinate
5. **Hard constraints are powerful** — reject prompts that violate safety thresholds

**Production recipe:**
```python
reward = (
    0.5 * task_completion +      # Does it solve the task?
    0.3 * accuracy +              # Is it correct?
    -0.5 * hallucination_rate +  # Severe penalty
    -0.2 * format_violations +   # Moderate penalty
    -0.1 * cost_overrun          # Small penalty
)
```

**Next Steps:**
1. **Notebook 03:** Add context features for smarter routing
2. **Your system:** Start logging hallucination flags immediately
3. **Monitoring:** Track hallucination rate over time (should decrease!)

**Golden rule:** If you can't measure it, don't optimize for it. Build hallucination detection first, then use it in your reward.